<a href="https://colab.research.google.com/github/UbiOneKenobi/titanic-ml-kaggle/blob/main/Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [111]:
import pandas as pd

train = pd.read_csv("/train.csv")
test = pd.read_csv("/test.csv")

To download the train and test .csv files visit: https://www.kaggle.com/competitions/titanic/data

### Variable	Definition
pclass:	Ticket class


sex:	Sex


Age:	Age in years


sibsp:	# of siblings / spouses aboard the Titanic



parch:	# of parents / children aboard the Titanic


ticket:	Ticket number


fare:	Passenger fare


cabin:	Cabin number



embarked:	Port of Embarkation	C = Cherbourg, Q = Queenstown, S = Southampton

In [112]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [113]:
train.info() #891 values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [114]:
train["Age"].describe()

,Age
count,714.000000
mean,29.699118
std,14.526497
min,0.420000
25%,20.125000
50%,28.000000
75%,38.000000
max,80.000000


In [115]:
test.info() #418 values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 36.1+ KB


In [116]:
corr_matrix = train.corr(numeric_only=True)
corr_matrix # the only usable linear correlation is between Fare and Survived: 0.257307

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
PassengerId,1.000000,-0.005007,-0.035144,0.036847,-0.057527,-0.001652,0.012658
Survived,-0.005007,1.000000,-0.338481,-0.077221,-0.035322,0.081629,0.257307
Pclass,-0.035144,-0.338481,1.000000,-0.369226,0.083081,0.018443,-0.549500
Age,0.036847,-0.077221,-0.369226,1.000000,-0.308247,-0.189119,0.096067
SibSp,-0.057527,-0.035322,0.083081,-0.308247,1.000000,0.414838,0.159651
Parch,-0.001652,0.081629,0.018443,-0.189119,0.414838,1.000000,0.216225
Fare,0.012658,0.257307,-0.549500,0.096067,0.159651,0.216225,1.000000


Let's now analyze the relationships between Survived and the qualitative variables such as "Sex".

To do so we can't simply ask a scatter matrix, but we can instead analyze the distribution of "Survived" between the different classes of each variable.

In [117]:
embarked_survived = train.groupby("Embarked")["Survived"].mean()
sex_survived = train.groupby("Sex")["Survived"].mean()
cabin_survived = train.groupby("Cabin")["Survived"].mean()



In [118]:
embarked_survived

,Survived
Embarked,
C,0.553571
Q,0.389610
S,0.336957


As we have seen from the results that, 55.4%, 39% and 33.7% of passengers from respectively Cherbourg, Queenstown and Southampton survived.

This is an interesting result that we definitely need to keep in mind.

In [119]:
sex_survived

,Survived
Sex,
female,0.742038
male,0.188908


In [120]:
cabin_survived

,Survived
Cabin,
A10,0.0
A14,0.0
A16,1.0
A19,0.0
A20,1.0
...,...
F33,1.0
F38,0.0
F4,1.0


## Feature Engineering

Create more informative features from the original passenger data:

- FamSize: total number of family members travelling together.
- Title: passenger title extracted from Name.
- Corridor: deck/area extracted from the first character of Cabin.
- TicketGroupSize: number of passengers sharing the same ticket.
"""

In [121]:
train["FamSize"] = train["SibSp"] + train["Parch"] + 1
test["FamSize"] = test["SibSp"] + test["Parch"] + 1

train["FamSize"].head()

,FamSize
0,2
1,2
2,1
3,2
4,1


In [122]:
train["FamSize"].value_counts()

,count
FamSize,
1,537
2,161
3,102
4,29
6,22
5,15
7,12
11,7
8,6


In [123]:
train["Title"] = train["Name"].str.split(",").str[1].str.split(".").str[0]
test["Title"] = test["Name"].str.split(",").str[1].str.split(".").str[0]

train["Title"].head()

,Title
0,Mr
1,Mrs
2,Miss
3,Mrs
4,Mr


In [124]:
train["Corridor"] = train["Cabin"].str[0]
test["Corridor"] = test["Cabin"].str[0]

train["Corridor"].head()

,Corridor
0,NaN
1,C
2,NaN
3,C
4,NaN


In [125]:
train["Corridor"].value_counts()

,count
Corridor,
C,59
B,47
D,33
E,32
A,15
F,13
G,4
T,1


TicketGroupSize captures how many passengers share the same ticket.

transform("count") is used instead of count() because we need the
group size mapped back to every original passenger row.

In [126]:
train["TicketGroupSize"] = train.groupby("Ticket")["Ticket"].transform("count")

test["TicketGroupSize"] = test.groupby("Ticket")["Ticket"].transform("count")

In [127]:
train[["TicketGroupSize", "Ticket"]].head()

,TicketGroupSize,Ticket
0,1,A/5 21171
1,1,PC 17599
2,1,STON/O2. 3101282
3,2,113803
4,1,373450


Let's now analyze the relationship of our new features with survived.

In [128]:
fam_size_survived = train.groupby("FamSize")["Survived"].mean()
title_survived = train.groupby("Title")["Survived"].mean()
corridor_survived = train.groupby("Corridor")["Survived"].mean()
ticket_survived = train.groupby("TicketGroupSize")["Survived"].mean()

In [129]:
fam_size_survived

,Survived
FamSize,
1,0.303538
2,0.552795
3,0.578431
4,0.724138
5,0.200000
6,0.136364
7,0.333333
8,0.000000
11,0.000000


In [130]:
title_survived

,Survived
Title,
Capt,0.000000
Col,0.500000
Don,0.000000
Dr,0.428571
Jonkheer,0.000000
Lady,1.000000
Major,0.500000
Master,0.575000
Miss,0.697802


In [131]:
corridor_survived

,Survived
Corridor,
A,0.466667
B,0.744681
C,0.593220
D,0.757576
E,0.750000
F,0.615385
G,0.500000
T,0.000000


In [132]:
ticket_survived

,Survived
TicketGroupSize,
1,0.297989
2,0.574468
3,0.698413
4,0.500000
5,0.000000
6,0.000000
7,0.238095


Let's now remove the old features that we don't use anymore in the training set and divide the label from the training data.

In [133]:
drop_train = ["SibSp", "Parch", "Cabin", "Ticket", "Name", "Survived"]
drop_test = ["SibSp", "Parch", "Cabin", "Ticket", "Name"]

X_train = train.drop(drop_train, axis =1)
X_test = test.drop(drop_test, axis =1)

X_train.head()

,PassengerId,Pclass,Sex,Age,Fare,Embarked,FamSize,Title,Corridor,TicketGroupSize
0,1,3,male,22.0,7.2500,S,2,Mr,NaN,1
1,2,1,female,38.0,71.2833,C,2,Mrs,C,1
2,3,3,female,26.0,7.9250,S,1,Miss,NaN,1
3,4,1,female,35.0,53.1000,S,2,Mrs,C,2
4,5,3,male,35.0,8.0500,S,1,Mr,NaN,1


In [134]:
y_train = train["Survived"].copy()
y_train.head()

,Survived
0,0
1,1
2,1
3,1
4,0


## Preprocessing

- Categorical features are encoded with OneHotEncoder.
- Missing Corridor values are treated as an "Unknown" category.
- Missing numerical values in Age and Fare are replaced with the median.
- Numerical features are standardized because distance-based models such as KNN
  are sensitive to feature scale.


In [135]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

corridor_pip = make_pipeline(SimpleImputer(strategy="constant",fill_value= "Unknown"), OneHotEncoder(handle_unknown="ignore"))
age_pip = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
fare_pip = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

In [136]:
cat_feat = ["Sex", "Embarked", "Title"]
num_feat = ["Pclass", "FamSize", "TicketGroupSize"]

preprocessing= ColumnTransformer([
    ("corr", corridor_pip, ["Corridor"]),
     ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feat),
      ("num", StandardScaler(), num_feat),
      ("age", age_pip, ["Age"]),
    ("fare", fare_pip, ["Fare"])
      ])

In [137]:
X_train_processed = preprocessing.fit_transform(X_train)
preprocessing.get_feature_names_out()

array(['corr__Corridor_A', 'corr__Corridor_B', 'corr__Corridor_C',
       'corr__Corridor_D', 'corr__Corridor_E', 'corr__Corridor_F',
       'corr__Corridor_G', 'corr__Corridor_T', 'corr__Corridor_Unknown',
       'cat__Sex_female', 'cat__Sex_male', 'cat__Embarked_C',
       'cat__Embarked_Q', 'cat__Embarked_S', 'cat__Embarked_nan',
       'cat__Title_ Capt', 'cat__Title_ Col', 'cat__Title_ Don',
       'cat__Title_ Dr', 'cat__Title_ Jonkheer', 'cat__Title_ Lady',
       'cat__Title_ Major', 'cat__Title_ Master', 'cat__Title_ Miss',
       'cat__Title_ Mlle', 'cat__Title_ Mme', 'cat__Title_ Mr',
       'cat__Title_ Mrs', 'cat__Title_ Ms', 'cat__Title_ Rev',
       'cat__Title_ Sir', 'cat__Title_ the Countess', 'num__Pclass',
       'num__FamSize', 'num__TicketGroupSize', 'age__Age', 'fare__Fare'],
      dtype=object)

Now that we have processed the features we can start the training of our model.


## Model Training — K-Nearest Neighbors

KNN is used as the first classification baseline.
The full preprocessing pipeline is included in cross-validation to avoid
fitting preprocessing transformations on validation folds.

In [138]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

knn_pip = make_pipeline(preprocessing, KNeighborsClassifier())

score = cross_val_score(knn_pip, X_train, y_train, cv=4, scoring="accuracy").mean()
score

np.float64(0.8170827778451096)

### KNN Hyperparameter Tuning

GridSearchCV tests different combinations of:
- n_neighbors: number of neighbors considered for prediction.
- weights: whether all neighbors contribute equally or closer neighbors receive more weight.

In [139]:
knn_pip.get_params().keys()

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'columntransformer', 'kneighborsclassifier', 'columntransformer__force_int_remainder_cols', 'columntransformer__n_jobs', 'columntransformer__remainder', 'columntransformer__sparse_threshold', 'columntransformer__transformer_weights', 'columntransformer__transformers', 'columntransformer__verbose', 'columntransformer__verbose_feature_names_out', 'columntransformer__corr', 'columntransformer__cat', 'columntransformer__num', 'columntransformer__age', 'columntransformer__fare', 'columntransformer__corr__memory', 'columntransformer__corr__steps', 'columntransformer__corr__transform_input', 'columntransformer__corr__verbose', 'columntransformer__corr__simpleimputer', 'columntransformer__corr__onehotencoder', 'columntransformer__corr__simpleimputer__add_indicator', 'columntransformer__corr__simpleimputer__copy', 'columntransformer__corr__simpleimputer__fill_value', 'columntransformer__corr__simpleimputer__keep_empty_features', 'colum

In [140]:
from sklearn.model_selection import GridSearchCV


param_grid = {
    'kneighborsclassifier__n_neighbors': [3, 4, 5, 6, 8, 10, 12, 14, 16, 20, 50],
     'kneighborsclassifier__weights': ["uniform", "distance"]
    }

grid_search_cv = GridSearchCV(knn_pip, param_grid, cv=3, scoring='accuracy')
grid_search_cv.fit(X_train, y_train)

grid_search_cv.best_params_

{'kneighborsclassifier__n_neighbors': 16,
 'kneighborsclassifier__weights': 'uniform'}

In [141]:
final_model = grid_search_cv.best_estimator_
score = cross_val_score(final_model, X_train, y_train, cv=4, scoring="accuracy").mean()
score

np.float64(0.8260665373894074)

KNN shows limited improvement after hyperparameter tuning,
so Logistic Regression is evaluated as an alternative classifier.

In [142]:
from sklearn.linear_model import LogisticRegression

logr_clf = LogisticRegression(random_state = 42)

logr_pip = make_pipeline(preprocessing, logr_clf)


score = cross_val_score(logr_pip, X_train, y_train, cv=3, scoring="accuracy").mean()
score.round(2)

np.float64(0.83)

In [143]:
logr_pip.get_params().keys()

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'columntransformer', 'logisticregression', 'columntransformer__force_int_remainder_cols', 'columntransformer__n_jobs', 'columntransformer__remainder', 'columntransformer__sparse_threshold', 'columntransformer__transformer_weights', 'columntransformer__transformers', 'columntransformer__verbose', 'columntransformer__verbose_feature_names_out', 'columntransformer__corr', 'columntransformer__cat', 'columntransformer__num', 'columntransformer__age', 'columntransformer__fare', 'columntransformer__corr__memory', 'columntransformer__corr__steps', 'columntransformer__corr__transform_input', 'columntransformer__corr__verbose', 'columntransformer__corr__simpleimputer', 'columntransformer__corr__onehotencoder', 'columntransformer__corr__simpleimputer__add_indicator', 'columntransformer__corr__simpleimputer__copy', 'columntransformer__corr__simpleimputer__fill_value', 'columntransformer__corr__simpleimputer__keep_empty_features', 'columnt

### Logistic Regression Hyperparameter Tuning

RandomizedSearchCV is used to explore different values of the regularization
parameter C. A log-uniform distribution is appropriate because useful values
of C may span several orders of magnitude.

In [144]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV


logr_clf = LogisticRegression(random_state = 42, solver="liblinear", penalty="l2")
logr_pip = make_pipeline(preprocessing, logr_clf)

param_distribs = {
    "logisticregression__C": loguniform(0.001, 100)
}

rnd_cv = RandomizedSearchCV(logr_pip, param_distributions=param_distribs, n_iter=10, cv=3, scoring='accuracy')
rnd_cv.fit(X_train, y_train)

rnd_cv.best_params_

{'logisticregression__C': np.float64(1.0698346573741866)}

In [145]:
final_model = rnd_cv.best_estimator_

## Final Model

Logistic Regression is selected as the final model based on its
cross-validation performance (~82%).

The selected pipeline is fitted on the full training set and then used
to generate predictions for the Kaggle test set.

In [146]:
final_model.fit(X_train, y_train)

predictions = final_model.predict(X_test)
predictions

array([0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1,
       1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1,
       1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1,
       0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,

## Kaggle Submission

Create the submission file using the PassengerId from the test set
and the predicted Survived labels.

In [147]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predictions
})

In [148]:
submission.to_csv("submission.csv", index=False)

In [149]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>